In [1]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 14.4 MB/s eta 0:00:00


In [6]:
import pandas as pd
import time
import os
from groq import Groq

# ─── Config ─────────────────────────────────────────────

PROMPT_COL   = "prompt"

# Models in priority order — if one hits rate limit, next is used
MODELS = [
    "llama-3.1-8b-instant",    # best balance — fast, small, good quality
    "llama3-8b-8192",          # fallback 1 — similar size, different limits
    "gemma2-9b-it",            # fallback 2 — slightly bigger, great quality
    "llama-3.2-3b-preview",    # fallback 3 — tiny, almost never rate limited
]
# ────────────────────────────────────────────────────────

client = Groq(api_key=GROQ_API_KEY)

# tracks which model we are currently using
current_model_index = 0

def enhance_prompt(bad_prompt):
    global current_model_index

    # try every model before giving up on this row
    attempts = 0

    while attempts < len(MODELS):
        model = MODELS[current_model_index]

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": """You are an expert prompt engineer.
Transform the given basic prompt into a high quality, detailed,
and effective prompt that gets the best possible AI response.

Rules:
- Keep the original intent and topic
- Add role, context, format, tone, and constraints
- Do NOT answer the prompt, only improve it
- Return ONLY the improved prompt, nothing else"""
                    },
                    {
                        "role": "user",
                        "content": f"Original prompt: {bad_prompt}\nImproved prompt:"
                    }
                ],
                temperature=0.7,
                max_tokens=512,
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            error = str(e).lower()
            error_codes = ["429", "400"]
            # ── Rate limit hit ───────────────────────────
            if "rate limit" in error or any(code in error for code in error_codes):
                print(f"  ⚠️  Rate limit hit on '{model}'")

                # shift to next model
                current_model_index = (current_model_index + 1) % len(MODELS)
                next_model = MODELS[current_model_index]
                print(f"  🔄 Switching to '{next_model}'")
                attempts += 1
                time.sleep(3)   # small pause before retrying with new model

            # ── Other error (network, timeout etc) ───────
            else:
                print(f"  ❌ Unexpected error on '{model}': {e}")
                time.sleep(10)
                attempts += 1

    # all models failed for this row
    print(f"  ❌ All models failed for this prompt. Skipping.")
    return None


def main():
    df = pd.read_csv(INPUT_CSV)
    print(f"Total prompts: {len(df)}")

    # ── Resume support ───────────────────────────────────
    results = []
    if os.path.exists(OUTPUT_CSV):
        done_df     = pd.read_csv(OUTPUT_CSV)
        done_inputs = set(done_df["input"].tolist())
        results     = done_df.to_dict("records")
        df          = df[~df[PROMPT_COL].isin(done_inputs)]
        print(f"Resuming — {len(done_inputs)} already done, {len(df)} remaining")
    # ─────────────────────────────────────────────────────

    failed = []

    for i, row in df.iterrows():
        bad_prompt = row[PROMPT_COL]
        print(f"[{i+1}/{len(df)}] Processing... (model: {MODELS[current_model_index]})")

        enhanced = enhance_prompt(bad_prompt)

        if enhanced:
            results.append({
                "input":  bad_prompt,
                "output": enhanced
            })
            print(f"  ✅ Done")
        else:
            failed.append(i)
            print(f"  ⚠️  Skipped row {i}")

        time.sleep(2)

        # Save every 100 rows
        if (i + 1) % 100 == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"\n  💾 Progress saved — {len(results)} rows done\n")

    # Final save
    final_df = pd.DataFrame(results)
    final_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\n🎉 Done!")
    print(f"✅ Successful : {len(results)}")
    print(f"❌ Failed     : {len(failed)}")
    print(f"💾 Saved to   : {OUTPUT_CSV}")
    if failed:
        print(f"⚠️  Failed indices: {failed}")


if __name__ == "__main__":
    main()

Total prompts: 3459
Resuming — 200 already done, 3259 remaining
[201/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[202/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[203/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[204/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[205/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[206/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[207/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[208/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[209/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[210/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[211/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[212/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[213/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[214/3259] Processing... (model: llama-3.1-8b-instant)
  ✅ Done
[215/3259] Processing... (model: llama-3

KeyboardInterrupt: 